<h2>Setup</h2>

In [2]:
import time

from pyvo.dal import TAPService

In [3]:
svc = TAPService("http://35.239.90.76/tap")

In [4]:
def query(sql, print_info=True):
    """Run a query, return the results as a DataFrame, get basic performance timing, and 
    print along with row count."""
    print(f"Executing query: {sql}")
    start = time.perf_counter()
    res = svc.run_async(sql).resultstable
    end = time.perf_counter()
    df = res.to_table().to_pandas()
    if print_info:
        print(f"Query took {end - start:.3f} seconds and returned {len(df)} rows")
    return df

<h2>TAP Schema</h2>

In [5]:
query("SELECT * FROM tap_schema.columns WHERE table_name LIKE 'ppdb_lsstcam.%'")

Executing query: SELECT * FROM tap_schema.columns WHERE table_name LIKE 'ppdb_lsstcam.%'
Query took 1.199 seconds and returned 196 rows


,table_name,column_name,utype,ucd,unit,description,datatype,arraysize,xtype,__size_,principal,indexed,std,column_index
0,ppdb_lsstcam.DiaObject,raErr,,stat.error;pos.eq.ra,deg,Uncertainty of ra.,double,,,<NA>,0,0,0,<NA>
1,ppdb_lsstcam.DiaObject,diaObjectId,,meta.id;src,,Unique identifier of this DiaObject.,long,,,<NA>,0,0,0,<NA>
2,ppdb_lsstcam.DiaObject,validityStartMjdTai,,time.epoch,,Processing time when validity of this diaObjec...,double,,,<NA>,0,1,0,<NA>
3,ppdb_lsstcam.DiaObject,validityEndMjdTai,,time.epoch,,Processing time when validity of this diaObjec...,double,,,<NA>,0,0,0,<NA>
4,ppdb_lsstcam.DiaObject,ra,,pos.eq.ra,deg,Right ascension coordinate of the position of ...,double,,,<NA>,0,0,0,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191,ppdb_lsstcam.DiaSource,reliability,,,,Probability (0-1) that the diaSource is astrop...,double,,,<NA>,0,0,0,<NA>
192,ppdb_lsstcam.DiaForcedSource,psfFlux,,phot.count,nJy,Point Source model flux.,double,,,<NA>,0,0,0,<NA>
193,ppdb_lsstcam.DiaForcedSource,psfFluxErr,,stat.error;phot.count,nJy,Uncertainty of psfFlux.,double,,,<NA>,0,0,0,<NA>
194,ppdb_lsstcam.DiaForcedSource,scienceFlux,,phot.count,nJy,Forced photometry flux for a point source mode...,double,,,<NA>,0,0,0,<NA>


<h2>ID</h2>

In [11]:
query("SELECT * FROM ppdb_lsstcam.DiaObject WHERE diaObjectId=169760231406961662")

Executing query: SELECT * FROM ppdb_lsstcam.DiaObject WHERE diaObjectId=169760231406961662
Query took 3.069 seconds and returned 1 rows


,raErr,diaObjectId,validityStartMjdTai,validityEndMjdTai,ra,dec,decErr,u_psfFluxNdata,g_psfFluxNdata,r_psfFluxNdata,...,i_psfFluxMaxSlope,i_psfFluxErrMean,z_psfFluxMin,z_psfFluxMax,z_psfFluxMaxSlope,z_psfFluxErrMean,y_psfFluxMin,y_psfFluxMax,y_psfFluxMaxSlope,y_psfFluxErrMean
0,0.000032,169760231406961662,61029.350145,NaN,224.394815,-41.726785,0.000043,0,0,0,...,NaN,NaN,7422.667969,7422.667969,NaN,1156.7052,NaN,NaN,NaN,NaN


<h2>Cone Search</h2>

Sample ra and dec

In [7]:
ra = 224.0
dec = -41.7

In [8]:
query(f"""
SELECT diaObjectId, ra, dec
FROM ppdb_lsstcam.DiaObject
WHERE CONTAINS(
POINT('ICRS', ra, dec),
CIRCLE('ICRS', {ra}, {dec}, 1.0)) = 1
""")

Executing query: 
SELECT diaObjectId, ra, dec
FROM ppdb_lsstcam.DiaObject
WHERE CONTAINS(
POINT('ICRS', ra, dec),
CIRCLE('ICRS', 224.0, -41.7, 1.0)) = 1

Query took 3.096 seconds and returned 2340 rows


,diaObjectId,ra,dec
0,169760231406961662,224.394815,-41.726785
1,169760231408533932,224.500520,-41.742354
2,169760231527023344,225.093822,-41.128192
3,169760231528071214,225.008683,-41.252520
4,169760231709999999,222.924284,-41.159184
...,...,...,...
2335,169760233482617227,224.672758,-40.923658
2336,169760235018256650,225.131825,-41.253196
2337,169760235201233612,223.042480,-41.373831
2338,169760235223253202,224.087783,-41.177389


<h2>Nearest Neighbor Search</h2>

In [9]:
query(f"""
SELECT 
    o1.ra as ra1,
    o1.dec as dec1,
    o2.ra as ra2,
    o2.dec as dec2,
    o1.diaObjectId AS id1,
    o2.diaObjectId AS id2,
    DISTANCE(POINT('ICRS', o1.ra, o1.dec), POINT('ICRS', o2.ra, o2.dec)) AS dist
FROM ppdb_lsstcam.DiaObject AS o1
JOIN ppdb_lsstcam.DiaObject AS o2
  ON o1.diaObjectId <> o2.diaObjectId
 AND o1.validityStartMjdTai = o2.validityStartMjdTai
WHERE CONTAINS(POINT('ICRS', o1.ra, o1.dec),
               CIRCLE('ICRS', {ra}, {dec}, 0.5)) = 1
  AND DISTANCE(POINT('ICRS', o1.ra, o1.dec),
               POINT('ICRS', o2.ra, o2.dec)) < 0.02;
""")

Executing query: 
SELECT 
    o1.ra as ra1,
    o1.dec as dec1,
    o2.ra as ra2,
    o2.dec as dec2,
    o1.diaObjectId AS id1,
    o2.diaObjectId AS id2,
    DISTANCE(POINT('ICRS', o1.ra, o1.dec), POINT('ICRS', o2.ra, o2.dec)) AS dist
FROM ppdb_lsstcam.DiaObject AS o1
JOIN ppdb_lsstcam.DiaObject AS o2
  ON o1.diaObjectId <> o2.diaObjectId
 AND o1.validityStartMjdTai = o2.validityStartMjdTai
WHERE CONTAINS(POINT('ICRS', o1.ra, o1.dec),
               CIRCLE('ICRS', 224.0, -41.7, 0.5)) = 1
  AND DISTANCE(POINT('ICRS', o1.ra, o1.dec),
               POINT('ICRS', o2.ra, o2.dec)) < 0.02;

Query took 3.165 seconds and returned 334 rows


,ra1,dec1,ra2,dec2,id1,id2,dist
0,223.912710,-41.744910,223.913495,-41.742102,169760233475801895,169760233475801890,0.002869
1,224.303597,-41.348616,224.313934,-41.331839,169760231735165099,169760231735165083,0.018485
2,224.149859,-41.561136,224.161956,-41.553072,169760236969132575,169760231734641150,0.012123
3,223.770488,-41.735614,223.772153,-41.748072,169760235220632091,169760231730971356,0.012520
4,223.792678,-41.736721,223.772153,-41.748072,169760235220632154,169760231730971356,0.019062
...,...,...,...,...,...,...,...
329,223.770488,-41.735614,223.775212,-41.752265,169760235220632091,169760235220632166,0.017019
330,223.788152,-41.742726,223.775212,-41.752265,169760231730971379,169760235220632166,0.013572
331,224.230959,-41.423449,224.220888,-41.418600,169760231735165162,169760231735165134,0.008975
332,224.220144,-41.401772,224.220888,-41.418600,169760231735165092,169760231735165134,0.016837


<h2>Join</h2>

In [13]:
query("""SELECT * FROM ppdb_lsstcam.DiaSource ds 
LEFT JOIN ppdb_lsstcam.DiaObject dob ON dob.diaObjectId = ds.diaObjectId
WHERE dob.diaObjectId=169760231406961662""")

Executing query: SELECT * FROM ppdb_lsstcam.DiaSource ds 
LEFT JOIN ppdb_lsstcam.DiaObject dob ON dob.diaObjectId = ds.diaObjectId
WHERE dob.diaObjectId=169760231406961662
Query took 3.074 seconds and returned 1 rows


,diaSourceId,visit,diaObjectId,ssObjectId,parentDiaSourceId,ssObjectReassocTimeMjdTai,midpointMjdTai,ra,dec,centroid_flag,...,i_psfFluxMaxSlope,i_psfFluxErrMean,z_psfFluxMin,z_psfFluxMax,z_psfFluxMaxSlope,z_psfFluxErrMean,y_psfFluxMin,y_psfFluxMax,y_psfFluxMaxSlope,y_psfFluxErrMean
0,169760231406961662,2025121900254,169760231406961662,<NA>,0,NaN,61029.34573,224.394815,-41.726785,False,...,NaN,NaN,7422.667969,7422.667969,NaN,1156.7052,NaN,NaN,NaN,NaN


<h2>Table scan</h2>

In [27]:
query("SELECT * FROM ppdb_lsstcam.DiaObject WHERE r_psfFluxMean BETWEEN 1090.0 and 1100.0")

Executing query: SELECT * FROM ppdb_lsstcam.DiaObject WHERE r_psfFluxMean BETWEEN 1090.0 and 1100.0
Query took 7.244 seconds and returned 442 rows


,raErr,diaObjectId,validityStartMjdTai,validityEndMjdTai,ra,dec,decErr,u_psfFluxNdata,g_psfFluxNdata,r_psfFluxNdata,...,i_psfFluxMaxSlope,i_psfFluxErrMean,z_psfFluxMin,z_psfFluxMax,z_psfFluxMaxSlope,z_psfFluxErrMean,y_psfFluxMin,y_psfFluxMax,y_psfFluxMaxSlope,y_psfFluxErrMean
0,0.000020,169834973884317828,61046.162203,NaN,62.221586,-48.507052,0.000021,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.000024,169834973636853795,61046.160884,NaN,51.580674,-27.269728,0.000025,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.000021,169834973821403226,61046.161820,NaN,60.443123,-48.508281,0.000041,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.000035,169579900902047813,61046.160873,NaN,51.829092,-27.148622,0.000029,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.000019,169834973878026293,61046.162096,NaN,63.112323,-48.911122,0.000031,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
437,0.000015,169641483601708138,61004.303980,NaN,61.380117,-48.830758,0.000028,0,0,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
438,0.000006,169641483774197799,61004.304330,NaN,62.874961,-48.867252,0.000011,0,0,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
439,0.000023,169641485000507403,61004.304522,NaN,62.458306,-47.817780,0.000029,0,0,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
440,0.000028,169579900751577172,61002.230995,NaN,52.522031,-27.724372,0.000037,0,0,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
raise RuntimeError("Comment me out to run scratch queries")

In [ ]:
tbl = query("SELECT diaObjectId FROM ppdb.DiaObject as diaobj WHERE diaobj.ra >= 186 and diaobj.ra < 187") 
tbl.head()

In [ ]:
query("SELECT COUNT(DISTINCT diaObjectId) FROM ppdb.DiaObject") 

In [ ]:
query("SELECT MIN(ra) AS ra_min, MAX(ra) AS ra_max, MIN(dec) AS dec_min, MAX(dec) AS dec_max FROM ppdb.DiaObject")

In [ ]:
query("SELECT * FROM ppdb.DiaSource WHERE diaObjectId=24624704620855420")

In [ ]:
query("SELECT * FROM ppdb.DiaForcedSource WHERE diaObjectId=24624704620855420")

In [ ]:
query("""SELECT * 
FROM ppdb.DiaObject 
WHERE CONTAINS(POINT('ICRS', ra, dec), 
CIRCLE('ICRS', 186.8, 7.0, 0.1)) = 1""")

In [ ]:
query("""SELECT * FROM ppdb.DiaSource ds 
LEFT JOIN ppdb.DiaObject dob ON dob.diaObjectId = ds.diaObjectId
WHERE dob.diaObjectId=169760231406961662
""")

In [ ]:
query("query("SELECT * FROM ppdb.DiaSource WHERE diaObjectId=24624704620855420")

In [ ]:
query("SELECT * FROM ppdb.DiaObject 
WHERE r_psfFluxMean BETWEEN 1090.0 and 1100.0")

In [ ]:
nn2_sql = """
SELECT TOP 1000 o1.diaObjectId AS id1, o2.diaObjectId AS id2,
       DISTANCE(POINT('ICRS', o1.ra, o1.dec), POINT('ICRS', o2.ra, o2.dec)) AS d
FROM   ppdb.DiaObject o1
JOIN   ppdb.DiaObject o2
  ON   o1.diaObjectId < o2.diaObjectId
WHERE  CONTAINS(POINT('ICRS', o1.ra, o1.dec),
                CIRCLE('ICRS', 186.5, 7.05, 0.5))=1
 AND   DISTANCE(POINT('ICRS', o1.ra, o1.dec),
                POINT('ICRS', o2.ra, o2.dec)) < 0.5;
"""
query(nn2_sql)

In [ ]:
query("SELECT diaObjectId, ra, dec FROM ppdb_lsstcam.DiaObject WHERE CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', 186.8, 7.0, 0.1)) = 1")

In [34]:
svc.search("SELECT diaObjectId FROM ppdb_lsstcam.DiaObject")

<DALResultsTable length=689808>
   diaObjectId    
      int64       
------------------
169747015279837233
169738329461884410
169298432603193361
169298432603193361
169298432603193361
169298432603193401
169298432608961411
169298432740032523
169298432740032523
               ...
169716270731624535
169720648292630606
169593100859605095
169738309537890312
169549125720735996
169663451070202462
169738329350209591
169738329618120722
169738329350209591
169738329888653369

<h2>DiaObject Counts by Day</h2>

In [23]:
query("""SELECT FLOOR(validityStartMjdTai) as mjd_tai_day, COUNT(*) as dia_objects
FROM ppdb_lsstcam.DiaObject
GROUP BY mjd_tai_day
ORDER BY mjd_tai_day DESC
""")

Executing query: SELECT FLOOR(validityStartMjdTai) as mjd_tai_day, COUNT(*) as dia_objects
FROM ppdb_lsstcam.DiaObject
GROUP BY mjd_tai_day
ORDER BY mjd_tai_day DESC

Query took 3.057 seconds and returned 45 rows


,mjd_tai_day,dia_objects
0,61047.0,2152
1,61046.0,12356
2,61045.0,7232
3,61041.0,211
4,61040.0,1088
5,61034.0,17351
6,61033.0,7373
7,61032.0,13338
8,61029.0,35565
9,61028.0,41955
